## XGBoost v. NN baseline

**Config**: n_train at 100k, n_test at 10k, $\sigma \in [0.1, 0.5],$ with $r, T$ default values. All hyperparameters at default.

This experiment determines the performance (RMSE, MAE) of both a neural network and a standard XGBoost model across log moneyness regions. The standard regions used are as follows:
- ITM: $lm = \log \left ( \frac{S}{K} \right ) \in [0.5, 2.0)$
- ATM: $lm \in [-0.2, 0.2]$
- OTM: $lm \in [-2, -0.5]$

In [1]:
import sys, os, torch

sys.path.append("..")
torch.set_num_threads(1)
torch.set_num_interop_threads(1)

from src.data.dataset_generator import DataConfig, generate_bundle

# data generation code
base_cfg = DataConfig(
    n_train=100_000,
    n_test=10_000
    )

regions = {
    "ATM": (-0.2, 0.2),
    "ITM": (0.5, 2.0),
    "OTM": (-2, -0.5)
}

bundles = {
}

for region, lm_range in regions.items():
    cfg = base_cfg.replace(log_moneyness_range=lm_range)
    bundles[region] = generate_bundle(cfg)

### Hyperparameter tuning

Small grid search on a standard-distribution bundle (`log_m ∈ [-2, 2]`). We pick the config with lowest validation MAE for each model, then reuse those winners across all regimes below.

- XGBoost grid: `max_depth ∈ {5, 7}`, `learning_rate ∈ {0.03, 0.1}`, `n_estimators ∈ {500, 1000}` — 8 configs
- NN grid: `lr ∈ {1e-3, 3e-4}`, `loss_alpha ∈ {0.2, 0.5, 0.8}` — 6 configs

We start with tuning the NN, then the XGB.

In [2]:
import itertools
import numpy as np
import pandas as pd

from src.data.dataset_generator import generate_bundle
from src.training.train_xgboost import train as xgb_train, predict as xgb_predict
from src.training.train_nn import train as nn_train, predict as nn_predict, set_seed, get_device
from src.eval.validate import compute_metrics


def val_mae(y_true, y_pred):
    return float(np.mean(np.abs(y_pred - y_true)))


tuning_cfg = base_cfg  # base_cfg has log_moneyness_range default = (-2, 2)
tuning_bundle = generate_bundle(tuning_cfg)
Xt, yt = tuning_bundle.train.X, tuning_bundle.train.y_call
Xv, yv = tuning_bundle.val.X,   tuning_bundle.val.y_call

device = get_device(use_cpu=True)
nn_grid = list(itertools.product([1e-3, 3e-4], [0.2, 0.5, 0.8]))
nn_trials = []
for lr, loss_alpha in nn_grid:
    print(f"Training NN: lr={lr}, loss_alpha={loss_alpha}", flush=True)

    set_seed(42)
    params = {"lr": lr, "loss_alpha": loss_alpha}
    model, np_ = nn_train(Xt, yt, Xv, yv, device=device, patience=10, epochs=200, **params)
    y_pred = nn_predict(model, Xv, np_, device="cpu")

    mae = val_mae(yv, y_pred)
    print(f"Done: val_mae={mae}", flush=True)

    nn_trials.append({**params, "val_mae": val_mae(yv, y_pred)})


nn_trials_df = pd.DataFrame(nn_trials).sort_values("val_mae").reset_index(drop=True)
best_nn = {k: v for k, v in nn_trials_df.iloc[0].items() if k != "val_mae"}
print("\nNN trials:")
print(nn_trials_df)
print("Best NN:", best_nn)

Training NN: lr=0.001, loss_alpha=0.2
Epoch 0, Training Loss: 0.76395655, Eval Loss: 0.63140208
Epoch 10, Training Loss: 0.20175771, Eval Loss: 0.18724304
Epoch 20, Training Loss: 0.09536084, Eval Loss: 0.08470836
Epoch 30, Training Loss: 0.03919489, Eval Loss: 0.03479074
Epoch 40, Training Loss: 0.02178775, Eval Loss: 0.02137734
Epoch 50, Training Loss: 0.01992108, Eval Loss: 0.01930111
Epoch 60, Training Loss: 0.01595609, Eval Loss: 0.01522826
Epoch 70, Training Loss: 0.01376292, Eval Loss: 0.01334116
Epoch 80, Training Loss: 0.01217403, Eval Loss: 0.01175066
Epoch 90, Training Loss: 0.01080853, Eval Loss: 0.01045436
Epoch 100, Training Loss: 0.00967469, Eval Loss: 0.00937037
Epoch 110, Training Loss: 0.00873666, Eval Loss: 0.00845344
Epoch 120, Training Loss: 0.00790127, Eval Loss: 0.00762978
Epoch 130, Training Loss: 0.00706511, Eval Loss: 0.00679498
Epoch 140, Training Loss: 0.00618219, Eval Loss: 0.00591128
Epoch 150, Training Loss: 0.00527892, Eval Loss: 0.00500985
Epoch 160, Tr

In [3]:
xgb_grid = list(itertools.product([5, 7], [0.03, 0.1], [500, 1000]))
xgb_trials = []
for max_depth, lr, n_estimators in xgb_grid:
    params = {"max_depth": max_depth, "learning_rate": lr, "n_estimators": n_estimators}
    model, np_ = xgb_train(Xt, yt, Xv, yv, **params)
    y_pred = xgb_predict(model, Xv, np_)
    xgb_trials.append({**params, "val_mae": val_mae(yv, y_pred)})


xgb_trials_df = pd.DataFrame(xgb_trials).sort_values("val_mae").reset_index(drop=True)
best_xgb = {k: v for k, v in xgb_trials_df.iloc[0].items() if k != "val_mae"}
best_xgb["max_depth"] = int(best_xgb["max_depth"])
best_xgb["n_estimators"] = int(best_xgb["n_estimators"])
print("XGBoost trials:")
print(xgb_trials_df)
print("Best XGBoost:", best_xgb)

[0]	validation_0-rmse:0.96827
[50]	validation_0-rmse:0.32610
[100]	validation_0-rmse:0.15791
[150]	validation_0-rmse:0.11417
[200]	validation_0-rmse:0.09836
[250]	validation_0-rmse:0.09342
[300]	validation_0-rmse:0.08811
[350]	validation_0-rmse:0.08540
[400]	validation_0-rmse:0.08305
[450]	validation_0-rmse:0.08024
[499]	validation_0-rmse:0.07725
[0]	validation_0-rmse:0.96827
[50]	validation_0-rmse:0.32610
[100]	validation_0-rmse:0.15791
[150]	validation_0-rmse:0.11417
[200]	validation_0-rmse:0.09836
[250]	validation_0-rmse:0.09342
[300]	validation_0-rmse:0.08811
[350]	validation_0-rmse:0.08540
[400]	validation_0-rmse:0.08305
[450]	validation_0-rmse:0.08024
[500]	validation_0-rmse:0.07725
[550]	validation_0-rmse:0.07620
[600]	validation_0-rmse:0.07441
[650]	validation_0-rmse:0.07336
[700]	validation_0-rmse:0.07252
[750]	validation_0-rmse:0.07129
[800]	validation_0-rmse:0.07070
[850]	validation_0-rmse:0.06977
[900]	validation_0-rmse:0.06809
[950]	validation_0-rmse:0.06696
[999]	validati

## Training and Eval

We train both models on the ranges mentioned earlier. Subsequently, we compare the model's prediction metrics across test data seeded from the same ranges as the training data:
- MAE
- RMSE
- Mean and median relative error
- Maximum error

Both models utilize similar feature engineering, opting for log-moneyness instead of standard $S$ and $K$ values. This is because option price $C$ increases roughly monotonically with $\log \left ( \frac{S}{K} \right ).$ For true predictions, we can mathematically compute log moneyness manually.
Furthermore, the NN uses a hybridized loss function, linearly interpolating standard MSE with a $y-$weighted MSE. This is done to minimize both relative error at OTM, and absolute error at ITM.

In [6]:
import pandas as pd
from src.training.train_nn import train, set_seed, get_device, predict
from src.eval.validate import preprocess_test_data, compute_metrics

set_seed(42) 
device = get_device(use_cpu=True)

results = []

for region in bundles:
    # Train only on calls, for now
    bundle = bundles[region]
    X_train, y_train = bundle.train.X, bundle.train.y_call
    X_val, y_val = bundle.val.X, bundle.val.y_call
    
    nn_model, norm_params = train(
        X_train, y_train, X_val, y_val,
        device=device,
        patience=25,
        epochs=400,
        **best_nn,
    )

    X_test, y_test = bundle.test.X, bundle.test.y_call
    X_test, _ = preprocess_test_data(X_test)
    y_pred = predict(nn_model, X_test, norm_params, device="cpu")

    metrics = compute_metrics(y_test, y_pred, min_value_threshold=0.01)
    results.append({
        "model" : "NN",
        "regime" : region,
        **{k: v for k, v in metrics.items() if k not in ["errors", "abs_errors", "relative_errors"]}
    })

Epoch 0, Training Loss: 0.41499999, Eval Loss: 0.34418595
Epoch 10, Training Loss: 0.16844915, Eval Loss: 0.15472235
Epoch 20, Training Loss: 0.12994777, Eval Loss: 0.12146574
Epoch 30, Training Loss: 0.08896638, Eval Loss: 0.08338039
Epoch 40, Training Loss: 0.05952799, Eval Loss: 0.05638277
Epoch 50, Training Loss: 0.04105159, Eval Loss: 0.03911158
Epoch 60, Training Loss: 0.03016506, Eval Loss: 0.02896336
Epoch 70, Training Loss: 0.02308095, Eval Loss: 0.02219861
Epoch 80, Training Loss: 0.01823540, Eval Loss: 0.01749242
Epoch 90, Training Loss: 0.01473728, Eval Loss: 0.01408578
Epoch 100, Training Loss: 0.01209632, Eval Loss: 0.01154687
Epoch 110, Training Loss: 0.01011479, Eval Loss: 0.00967043
Epoch 120, Training Loss: 0.00863374, Eval Loss: 0.00828686
Epoch 130, Training Loss: 0.00750559, Eval Loss: 0.00722916
Epoch 140, Training Loss: 0.00662847, Eval Loss: 0.00640313
Epoch 150, Training Loss: 0.00595863, Eval Loss: 0.00575031
Epoch 160, Training Loss: 0.00539766, Eval Loss: 0.

Now to train the XGBoost model:

In [7]:
import pandas as pd
from src.training.train_xgboost import train as xgb_train, predict as xgb_predict
from src.eval.validate import preprocess_test_data, compute_metrics

xgb_results = []

for region in bundles:
    bundle = bundles[region]
    X_train, y_train = bundle.train.X, bundle.train.y_call
    X_val, y_val = bundle.val.X, bundle.val.y_call

    xgb_model, norm_params = xgb_train(
        X_train, y_train, 
        X_val, y_val, 
        **best_xgb
    )

    X_test_raw, y_test = bundle.test.X, bundle.test.y_call
    X_test, _ = preprocess_test_data(X_test_raw)
    y_pred = xgb_predict(xgb_model, X_test, norm_params)

    metrics = compute_metrics(y_test, y_pred, min_value_threshold=0.01)
    xgb_results.append({
        "model": "XGBoost",
        "regime": region,
        **{k: v for k, v in metrics.items() if k not in ["errors", "abs_errors", "relative_errors"]}
    })

[0]	validation_0-rmse:0.90488
[50]	validation_0-rmse:0.08244
[100]	validation_0-rmse:0.04779
[150]	validation_0-rmse:0.04131
[200]	validation_0-rmse:0.03568
[250]	validation_0-rmse:0.03353
[300]	validation_0-rmse:0.03124
[350]	validation_0-rmse:0.03000
[400]	validation_0-rmse:0.02853
[450]	validation_0-rmse:0.02752
[500]	validation_0-rmse:0.02667
[550]	validation_0-rmse:0.02625
[600]	validation_0-rmse:0.02577
[650]	validation_0-rmse:0.02543
[700]	validation_0-rmse:0.02513
[750]	validation_0-rmse:0.02485
[800]	validation_0-rmse:0.02463
[850]	validation_0-rmse:0.02440
[900]	validation_0-rmse:0.02406
[950]	validation_0-rmse:0.02389
[999]	validation_0-rmse:0.02370
[0]	validation_0-rmse:0.89651
[50]	validation_0-rmse:0.01508
[100]	validation_0-rmse:0.00567
[150]	validation_0-rmse:0.00539
[200]	validation_0-rmse:0.00526
[250]	validation_0-rmse:0.00518
[300]	validation_0-rmse:0.00512
[350]	validation_0-rmse:0.00506
[400]	validation_0-rmse:0.00502
[450]	validation_0-rmse:0.00498
[500]	validati

Finally, let's compare both!

In [8]:
results_df = pd.DataFrame(results)
xgb_results_df = pd.DataFrame(xgb_results)

print(pd.concat([results_df, xgb_results_df], ignore_index=True))

     model regime       mae      rmse  mean_relative_error  \
0       NN    ATM  0.003105  0.004949             0.028897   
1       NN    ITM  0.026709  0.036752             0.011826   
2       NN    OTM  0.000156  0.001057             0.085648   
3  XGBoost    ATM  0.002288  0.004002             0.021383   
4  XGBoost    ITM  0.006333  0.008275             0.002409   
5  XGBoost    OTM  0.000121  0.000854             0.073270   

   median_relative_error  max_error  
0               0.020443   0.033006  
1               0.008121   0.232231  
2               0.075003   0.039243  
3               0.013791   0.053130  
4               0.002142   0.064701  
5               0.049568   0.036489  


We observe that the NN and XGBoost predictions are not meaningfully different across ATM and OTM. **However, at ITM there exists around a 4.3x difference in MAE and RMSE**. This discrepancy most likely arises due to the inherent structure of the models rather than feature engineering. 

At ITM (or $S >> K$), both $D_1, D_2,$ grow very large. Thus, we approximate: $N(D_1), N(D_2) \approx 1,$ which means that
$$C = S - K e ^ {-rT}.$$ 
In our feature space $\log{\frac{C}{K}},$ this becomes low-curvature.

And although each model is trained independently within each regime, which eliminates cross-regime generalization effects, the aforementioned error persists, suggesting a structural bias within each model. Although GBTs also have this issue, the piecewise nature of its implementation means that each leaf within the model can be arbitrarily accurate, thus allowing them to reduce the structure by region and capture localized structure, in contrast to the NN, which is forced to minimize a global error. Furthermore, the exponential denormalization from feature-space $\log{\frac{C}{K}},$ to evaluation space $\frac{C}{K}$ punishes ITM prices disproportionately, thus amplifying the issue.

This is most likely why traditional implementations of Black-Scholes pricing often resort to GBTs over NNs for approximation.